# Pick My Movie -- Rapport final

## GRP 1 : Liliane Aroule, Zélie Delloye et Thibault Royer

### Sommaire
- Présentation du projet
  - Objectif et solutions
  - Dataset
- Les 3 modèles
  - Modèle de similarité utilisateurs
  - Modèle d'humeur de l'utilisateur
  - Modèle de similarité de Jacard
- Projet final
  - Environnement, Installation et librairie
  - Mise en place du projet
    - Chargement des données
    - Entraînement Modèle 1
    - Préparation Modèle 2
    - Modèle 3
  - Interface utilisateur
  - Utilisation des 3 modèles
  - Améliorations envisagées
- Liens Github
- Conclusion

### Présentation du projet

#### Objectif et solution

L'objectif de notre projet est de pouvoir donner des recommandations de films à un utilisateur grâce à un système combinant plusieurs modèles d'IA. Dans un premier temps, l'IA pose des questions sur le mood de l'utilisateur, ses envies actuelles, afin de présélectionner un ensemble de films qui lui correspondent.
Dans un second temps, deux modèles tournent conjointement sur ces films retenus : la similarité de Jaccard et la similarité cosinus. Ces deux modèles permettent de raffiner de plus en plus les recommandations en fonction du profil de l'utilisateur.
Enfin, si l'utilisateur le souhaite, un filtrage collaboratif est activé. Ce dernier se base sur les autres utilisateurs qui ont aimé les mêmes films pour affiner encore davantage les recommandations. Le but est que l'IA recommande des films qui plairaient de plus en plus à l'utilisateur.


#### Dataset
Notre dataset provient du lien suivant : https://grouplens.org/datasets/movielens/.
Movielens est une base de donnée créé par l'université du Minnesota, elle regroupe des films avec leur genre, leur titre, leurs années de sortie, et les notes attribués à chacun d'entre eux.

- ratings.csv : Contient les notes attribuées par les utilisateurs aux films.
- movies.csv : Contient les informations sur les films.
- genome-scores.csv : Contient la pertinence de chaque tag pour chaque film.
- genome-tags.csv : Associe chaque identifiant de tag à un mot-clé.
- links.csv : Associe les films à des bases de données externes.
- tags.csv : Contient des tags ajoutés librement par les utilisateurs.

### Les 3 modèles

#### Modèle collaboratif basé sur les utilisateurs

##### - Concept

Dans ce modèle, l'idée est de recommander des films à l'utilisateur en utilisant les autres utilisateurs qui ont des goûts similaires. L'idée principale est que deux utilisateurs ayant attribué des notes proches à certains films auront probablement des préférences communes.

##### - Fonctionnement

On commence par construire un profil utilisateur en faisant la moyenne des vecteurs de tags des films qu'il a aimés, cela donne une sorte de résumé des goûts de l'utilisateur. On compare ensuite ce profil à chaque film de la dataset via la similarité cosinus, qui mesure l'angle entre deux vecteurs en utilisant cette formule :
$$
\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}
$$
- Si la similarité est proche de 1 → utilisateurs très similaires  
- Si elle est proche de 0 → utilisateurs différents
- 
En parallèle, on construit une matrice utilisateurs × films à partir des notes. On se limite aux 3 000 utilisateurs avec le plus de note pour éviter de faire planter le Kernel, on identifie les 10 profils les plus proches puis on calcule la note moyenne qu'ils ont attribuée à chaque film, nous avons ainsi notre score collaboratif.
Les deux scores sont ensuite combinés :
$$\text{score\_final}(f) = 0{,}6 \times \text{score\_content}(f) + 0{,}4 \times \text{score\_collab}(f)$$
Les films déjà vus sont exclus du résultat final.


In [ ]:
top_users       = ratings_small["userId"].value_counts().head(3000).index
ratings_reduced = ratings_small[ratings_small["userId"].isin(top_users)]

user_movie_matrix = ratings_reduced.pivot_table(
    index="userId", columns="movieId", values="rating"
).fillna(0)

user_similarity_matrix = cosine_similarity(user_movie_matrix)
user_similarity_df     = pd.DataFrame(
    user_similarity_matrix,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

def score_collaboratif(liked_movie_ids, top_n_users=10):
    """
    Crée un pseudo-utilisateur à partir des films likés,
    trouve les utilisateurs les plus proches,
    retourne un score collaboratif par film.
    """
    # Vecteur du pseudo-utilisateur : 1 pour les films likés, 0 sinon
    pseudo_vector = pd.Series(0.0, index=user_movie_matrix.columns)
    for mid in liked_movie_ids:
        if mid in pseudo_vector.index:
            pseudo_vector[mid] = 5.0  # note max simulée

    # Similarité cosinus entre pseudo-user et tous les vrais users
    pseudo_arr   = pseudo_vector.values.reshape(1, -1)
    matrix_arr   = user_movie_matrix.values
    sims         = cosine_similarity(pseudo_arr, matrix_arr).flatten()
    sim_series   = pd.Series(sims, index=user_movie_matrix.index)
    similar_uids = sim_series.sort_values(ascending=False).head(top_n_users).index
    
    # Score moyen des utilisateurs similaires
    similar_ratings = ratings_small[ratings_small["userId"].isin(similar_uids)]
    raw_scores      = similar_ratings.groupby("movieId")["rating"].mean()
    popularity      = ratings_small.groupby("movieId").size()
    collab_scores   = (raw_scores / popularity).fillna(0)
    return collab_scores

#### Modèle de recherche sémantique basée sur l'humeur de l'utilisateur

##### - Concept

Ce modèle utilise du Machine Learning non supervisé en s'appuyant sur un modèle déjà entraîné : FastText. L'idée est d'associer l'humeur de l'utilisateur à des tags sémantiques, puis de retrouver les films dont les tags correspondent le mieux à ces mots-clés.

##### - Fonctionnement

Chaque tag du genome-tags.csv est transformé en vecteur numérique via FastText. C'est un système de word embeddings (représentation vectorielle des mots) qui va créer des vecteurs de dimension 300 de chaque tag.

Lorsque l'utilisateur choisit un mood, les mots-clés associés sont vectorisés dans ce même espace. On calcule leur similarité cosinus avec tous les tags du corpus pour identifier les plus proches. Même si le mot donné par l'utilisateur n'est pas forcément dans la base de données, FastText est capable de la comprendre et de le comparer à d'autres mots grâce à des subwords (n-grammes de caractères).

Pour chaque tag retenu, on va lui donner un certain poid en fonction de son importance, puis on récupère les films associés dans genome-scores.csv et on moyenne leurs scores de pertinence pour obtenir un classement final.

#### - Améliorations par rapport à la version intermédiaire

Nous avons décidé de changer de modèle de Machine Learning par rapport à la version intermédiaire.

En effet, nous avons d'abord voulu combiner l'utilisation de TF-IDF et SVD avec WordNet pour créer des vecteurs numériques des mots, puis les réduire, et enfin trouver des synonymes pour une recherche sémantique plus efficace.

Le problème de ce modèle était que si l'utilisateur donnait en entrée un mot inconnu dans la base de donnée, la sortie ne fonctionnait pas toujours car le modèle n'était pas assez robuste.

Le présent modèle, FastText, est bien plus robuste et combine la création de vecteur, la richesse de dimension et la richesse de la recherche sémantique à lui tout seul. Par contre il faut télécharger le modèle au préalable qui est plus lourd.
Pour avoir le modèle complet, il fallait télécharger un fichier de 7Gb, mais pour notre projet la version de gensim.downloader,  qui est moins lourde, suffit largement et fonctionne très bien.

In [4]:
def get_vector(text, model):
    
    if not isinstance(text, str): # sécurité
        return np.zeros(model.vector_size)
    words = text.lower().split()
    vectors = []
    
    for word in words:
        if word in model:
            vectors.append(model[word])

    if len(vectors) == 0: # si aucun mot connu
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)

def find_similar_tags(input_text, tags, tag_vectors, model, top_n=5):
    
    input_list = [w.strip().lower() for w in input_text.split(",")] # split des termes de l'utilisateur
    results = []
    for word in input_list:
        if word == "": # sécurité
            continue
        if word in tags: # mot est un tag existant
            idx = tags.index(word)
            word_vec = tag_vectors[idx]
        else:
            word_vec = get_vector(word, model) # mot inconnu : utilisation de FastText

        similarities = cosine_similarity(  # similarité avec tous les tags
            word_vec.reshape(1, -1),
            tag_vectors
        )[0]

        top_idx = np.argsort(similarities)[::-1][:top_n]

        for i in top_idx:
            tag = tags[i]
            score = similarities[i] # * 100
            results.append((tag, round(score, 2)))

    return np.array(results, dtype=object)

def find_similar_movies_from_tags(similar_tags, df, top_n=100):

    tag_weights = {tag: score for tag, score in similar_tags} # poids utilisateur

    df_filtered = df[df["tag"].isin(tag_weights.keys())].copy() # filtrer les tags utiles
    if df_filtered.empty:
        return np.array([], dtype=object)

    df_filtered["user_weight"] = df_filtered["tag"].map(tag_weights) # appliquer poids utilisateur

    df_filtered["weighted_score"] = df_filtered["relevance"] * df_filtered["user_weight"] # combiner les scores

    grouped = df_filtered.groupby(["movieId", "title"]) # agrégation par film

    movie_scores = grouped.apply(
        lambda x: x["weighted_score"].sum() / x["user_weight"].sum()
    )

    top_movies = movie_scores.sort_values(ascending=False).head(top_n) # top films

    result = np.array([ # sortie
        (title, round(score * 100, 2))
        for (movie_id, title), score in top_movies.items()
    ], dtype=object)

    return result

tags_df = pd.read_csv("genome-tags.csv") # chargement des données
scores_df = pd.read_csv("genome-scores.csv")
movies_df = pd.read_csv("movies.csv")

tags = tags_df["tag"].astype(str).tolist() # récupération de la liste des tags

tag_vectors = np.array([get_vector(tag, model) for tag in tags]) # création des vecteurs des tags

tag_vectors = normalize(tag_vectors) # normalisation des vecteurs

df = scores_df.merge(tags_df, on="tagId") # fusion des datasets
df = df.merge(movies_df, on="movieId")

similar_tags = find_similar_tags("wizard, magic",  tags, tag_vectors, model)
movies = find_similar_movies_from_tags(similar_tags, df)
print(movies)

[['Oz the Great and Powerful (2013)' 68.77]
 ['Seventh Son (2014)' 66.69]
 ['Stardust (2007)' 65.03]
 ['The Last Witch Hunter (2015)' 62.69]
 ['Brothers Grimm, The (2005)' 60.85]
 ['Willow (1988)' 60.61]
 ["Howl's Moving Castle (Hauru no ugoku shiro) (2004)" 58.91]
 ['Fantastic Beasts and Where to Find Them (2016)' 57.69]
 ['Witches, The (1990)' 57.19]
 ['Harry Potter and the Order of the Phoenix (2007)' 56.02]
 ['Solomon Kane (2009)' 55.71]
 ['Harry Potter and the Prisoner of Azkaban (2004)' 55.59]
 ['Harry Potter and the Chamber of Secrets (2002)' 54.87]
 ['Harry Potter and the Goblet of Fire (2005)' 54.65]
 ["Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)"
  54.6]
 ['Warlock (1989)' 54.51]
 ['Season of the Witch (2011)' 54.43]
 ['Troll (1986)' 54.42]
 ['Legend (1985)' 54.2]
 ['Practical Magic (1998)' 54.09]
 ['Dark Crystal, The (1982)' 53.53]
 ['Dragonslayer (1981)' 53.29]
 ['Covenant, The (2006)' 53.25]
 ['47 Ronin (2013)' 52.98]
 ['H

#### Modèle de similarité de Jaccard

##### - Concept

Ce modèle consiste à permettre à l'utilisateur de rentrer un nom de film qu'il aime. On utilise une méthode de recommandation qui propose des films similaires à ceux qu'il a déjà appréciés, en se basant sur les caractéristiques propres des films : le genre, l'année et la note moyenne, il s'agit donc d'un content-based filtering.

##### - Fonctionnement 
Pour deux ensembles (A) et (B), la similarité de Jaccard est :
$$
\text{Jaccard}(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$
- ∣A∩B∣ : nombre d’éléments communs
- ∣A∪B∣ : nombre total d’éléments uniques dans les deux ensembles

Le résultat est compris entre 0 et 1 :
- 0 → pas de genres en commun
- 1 → tous les genres sont identiques

##### - Score final

Pour un ensemble de films de référence $F_{ref}$, le score d'un film candidat $c$ est la moyenne de ses similarités avec chaque film de référence. Seuls les films ayant reçu un minimum de 20 votes sont considérés, afin d'écarter les films trop peu notés.

In [ ]:
def movie_similarity_jaccard(movie1, movie2, w_genre=0.5, w_rating=0.3, w_year=0.2):
    """Similarité Jaccard pondérée entre deux films."""
    genres1    = set(movie1["genres"].split("|")) if isinstance(movie1["genres"], str) else set()
    genres2    = set(movie2["genres"].split("|")) if isinstance(movie2["genres"], str) else set()
    genre_sim  = (len(genres1 & genres2) / len(genres1 | genres2)
                  if (genres1 or genres2) else 0)

    rating_sim = 1 - abs(movie1["mean_rating"] - movie2["mean_rating"]) / 5

    if np.isnan(movie1["year"]) or np.isnan(movie2["year"]):
        year_sim = 0
    else:
        year_sim = max(0, 1 - abs(movie1["year"] - movie2["year"]) / 50)

    return w_genre * genre_sim + w_rating * rating_sim + w_year * year_sim


def score_jaccard(liked_titles_clean, candidate_titles, min_votes=20):
    """
    Pour chaque film candidat, calcule la similarité Jaccard moyenne
    avec les films likés par l'utilisateur.
    Retourne un Series {title_clean: score}.
    """
    scores = {}
    liked_stats = []
    for t in liked_titles_clean:
        if t in movie_stats.index:
            liked_stats.append(movie_stats.loc[t])

    if not liked_stats:
        return pd.Series(dtype=float)
    for candidate in candidate_titles:
        if candidate not in movie_stats.index:
            continue
        cand_stat = movie_stats.loc[candidate]
        if cand_stat["num_ratings"] < min_votes:
            continue
        sim_list = [movie_similarity_jaccard(liked, cand_stat) for liked in liked_stats]
        scores[candidate] = np.mean(sim_list)

    return pd.Series(scores)

### Projet final

#### Environnement, Installation et librairie
Le projet s'exécute en Python 3.9+ dans un environnement Jupyter Notebook. Les fichiers de données movies.csv, ratings.csv, genome-scores.csv, genome-tags.csv, sont placés dans le même répertoire que le notebook.

Nous importons dans un premier temps, les librairies et les bibliothèques nécessaires pour faire fonctionner les 3 modèles :

Librairies :
- fasttext (via gensim.downloader) : convertit les tags en vecteurs numériques intelligents
- cosine_similarity : mesure de similarité entre utilisateurs et films

Bibilothèques :
- pandas : manipulation des données
- numpy : calculs numériques
- scikit-learn : calcul de similarité
- matplotlib : visualisation des résultats (Top 10 final)
- warnings
- re

On télécharge le modèle FastText via gensim et on le charge.

In [1]:
!pip install gensim
import pandas as pd
import numpy as np
import re
import warnings

from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

import gensim.downloader as api
model = api.load("fasttext-wiki-news-subwords-300")

#### Mise en place du projet

##### Chargement et nettoyage des données

On charge les 4 fichiers CSV et on nettoie les titres (minuscules, suppression de l'année entre parenthèses) et les notes puis on filtre les films populaires (≥ 50 notes) pour garantir la fiabilité des scores, et on construit les statistiques par film utilisées par le modèle de Jaccard. On créé les vecteurs films et tags pour le modèle collaboratif et on combine les génomes pour le modèles des moods.

In [ ]:
movies        = pd.read_csv("movies.csv")
ratings       = pd.read_csv("ratings.csv", nrows=200000)
genome_scores = pd.read_csv("genome-scores.csv")
genome_tags   = pd.read_csv("genome-tags.csv")

movies["year"]        = movies["title"].str.extract(r'\((\d{4})\)').astype(float)
movies["title_clean"] = (movies["title"].str.lower()
                          .str.replace(r"\(.*?\)", "", regex=True)
                          .str.strip())

ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")

df_merged   = pd.merge(ratings, movies, on="movieId") #Pour construire Jaccard
movie_stats = df_merged.groupby("title_clean").agg(
    mean_rating=("rating", "mean"),
    num_ratings=("rating", "count"),
    year=("year", "first"),
    genres=("genres", "first")

movie_counts   = ratings["movieId"].value_counts()
popular_movies = movie_counts[movie_counts >= 50].index
ratings_small  = ratings[ratings["movieId"].isin(popular_movies)]

In [ ]:
df_merged   = pd.merge(ratings, movies, on="movieId") #Pour construire Jaccard
movie_stats = df_merged.groupby("title_clean").agg(
    mean_rating=("rating", "mean"),
    num_ratings=("rating", "count"),
    year=("year", "first"),
    genres=("genres", "first")

movie_counts   = ratings["movieId"].value_counts() #filtrage des films populaires
popular_movies = movie_counts[movie_counts >= 50].index
ratings_small  = ratings[ratings["movieId"].isin(popular_movies)]

movie_vectors = genome_scores.pivot_table( #Création des vecteurs films et tags pour le modèle collaboratif
    index="movieId", columns="tagId", values="relevance"
).fillna(0)
movie_vectors = movie_vectors.loc[movie_vectors.index.intersection(ratings_small["movieId"])]

genome = genome_scores.merge(genome_tags, on="tagId").merge(movies[["movieId", "title"]], on="movieId") #Combinaison des genomes pour modèles des moods

##### Entraînement Modèle des tags : FastText
On vectorise les tags du genome avec le modèle FastText préalablement téléchargé.

In [ ]:
def get_vector(text, model):
    
    if not isinstance(text, str): # sécurité
        return np.zeros(model.vector_size)
    words = text.lower().split()
    vectors = []
    for word in words:
        if word in model:
            vectors.append(model[word])

    if len(vectors) == 0: # si aucun mot connu
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)

Pour chaque mot-clé donné par l'utilisateur, on trouve les tags de la base de données qui ont e plus de compatibilité pour enrichir la couverture sémantique de la recherche et la précision. 

In [ ]:
def find_similar_tags(input_text, tags, tag_vectors, model, top_n=5):

    
    input_list = [w.strip().lower() for w in input_text.split(",")] # split des termes de l'utilisateur
    results = []
    for word in input_list:
        if word == "":  # sécurité
            continue
        if word in tags:  # mot est un tag existant
            idx = tags.index(word)
            word_vec = tag_vectors[idx]
        else: # mot inconnu : utilisation de FastText
            word_vec = get_vector(word, model)

        similarities = cosine_similarity( # similarité avec tous les tags
            word_vec.reshape(1, -1),
            tag_vectors
        )[0]
        top_idx = np.argsort(similarities)[::-1][:top_n]

        for i in top_idx:
            tag = tags[i]
            score = similarities[i]
            results.append((tag, round(score, 2)))

    return np.array(results, dtype=object)

Puis, on calcule les moyennes de similarités de tous les tags pour chaque film. On donne des poids pour les tags données par l'utilisateur. On récupère les 100 meilleures moyennes en sortie.

In [ ]:
def find_similar_movies_from_tags(similar_tags, df, top_n=100):

    tag_weights = {tag: score for tag, score in similar_tags} # poids utilisateur
    
    df_filtered = df[df["tag"].isin(tag_weights.keys())].copy() # filtrer les tags utiles
    if df_filtered.empty:
        return np.array([], dtype=object)
    
    df_filtered["user_weight"] = df_filtered["tag"].map(tag_weights) # appliquer poids utilisateur
    
    df_filtered["weighted_score"] = df_filtered["relevance"] * df_filtered["user_weight"] # combiner les scores
    
    grouped = df_filtered.groupby(["movieId", "title"]) # agrégation par film
    movie_scores = grouped.apply(
        lambda x: x["weighted_score"].sum() / x["user_weight"].sum())

    top_movies = movie_scores.sort_values(ascending=False).head(top_n) # top films
    
    result = np.array([ # sortie
        (title, round(score * 100, 2))
        for (movie_id, title), score in top_movies.items()
    ], dtype=object)

    return result

##### Préparation Modèle collaboratif : Similarité Cosinus des Utilisateurs

On construit la matrice utilisateurs × films sur les 3 000 utilisateurs les plus actifs

In [ ]:
top_users         = ratings_small["userId"].value_counts().head(3000).index
user_movie_matrix = ratings_reduced.pivot_table(
    index="userId", columns="movieId", values="rating"
).fillna(0

On crée un pseudo-utilisateur à partir des films aimés (note simulée à 5), puis on calcule sa similarité cosinus avec tous les vrais utilisateurs pour identifier les profils proches :

In [ ]:
def score_collaboratif(liked_movie_ids, top_n_users=10):
    pseudo_vector = pd.Series(0.0, index=user_movie_matrix.columns)
    for mid in liked_movie_ids:
        if mid in pseudo_vector.index:
            pseudo_vector[mid] = 5.0  # note max simulée

    sims         = cosine_similarity(pseudo_vector.values.reshape(1, -1),
                                     user_movie_matrix.values).flatten()
    similar_uids = pd.Series(sims, index=user_movie_matrix.index)\
                     .sort_values(ascending=False).head(top_n_users).index

    collab_scores = ratings_small[ratings_small["userId"].isin(similar_uids)]\
                      .groupby("movieId")["rating"].mean()
    return collab_scores

##### Modèle des similarités de genre : Similarité de Jaccard

Pour chaque film candidat, on calcule une similarité pondérée avec les films aimés selon 3 critères 

In [ ]:
def movie_similarity_jaccard(movie1, movie2, w_genre=0.5, w_rating=0.3, w_year=0.2):
    genres1, genres2 = set(movie1["genres"].split("|")), set(movie2["genres"].split("|"))
    genre_sim  = len(genres1 & genres2) / len(genres1 | genres2)

    rating_sim = 1 - abs(movie1["mean_rating"] - movie2["mean_rating"]) / 5

    year_sim   = max(0, 1 - abs(movie1["year"] - movie2["year"]) / 50)

    return w_genre * genre_sim + w_rating * rating_sim + w_year * year_sim

On calcule le score moyen de chaque candidat face à tous les films aimés, en excluant les films avec moins de 20 votes

In [ ]:
def score_jaccard(liked_titles_clean, candidate_titles, min_votes=20):
    for candidate in candidate_titles:
        if movie_stats.loc[candidate]["num_ratings"] < min_votes:
            continue
        sim_list = [movie_similarity_jaccard(liked, movie_stats.loc[candidate])
                    for liked in liked_stats]
        scores[candidate] = np.mean(sim_list)
    return pd.Series(scores)

#### Interface utilisateur

L’interface utilisateur de notre application a été développée en Python à l’aide de la bibliothèque "CustomTkinter".

Cette interface permet à l’utilisateur d’interagir avec notre système de recommandation de films de manière intuitive, en guidant celui-ci à travers plusieurs étapes : saisie d’informations, sélection de préférences et visualisation des recommandations. (tout en utilisant nos 3 modèles successivement)

##### Architecture générale de l’interface

Notre interface est structurée en plusieurs “pages”, implémentées à l’aide de composants appelés frames.
Chaque frame correspond à une étape du parcours utilisateur :

1. saisie du nom
2. sélection de films déjà vus
3. choix d’utilisation du profil
4. sélection du mood
5. choix de films recommandés
6. affichage du résultat final

Nous avons fais en sorte d'obtenir un résultat proche d'une application "moderne". Le but étant de donner envie à l'utilisateur de continuer à se servir de notre système de recommandation.

##### Explication détaillée de l'interface

Installation de la bibliothèque :

In [3]:
pip install customtkinter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   -------------------- ------------------- 1/2 [customtkinter]
   -------------------- ------------------- 1/2 [customtkinter]
   -------------------- ------------------- 1/2 [customtkinter]
   -------------------- ------------------- 1/2 [customtkinter]
   -------------------- ------------------- 1/2 [customtkinter]
   ---------------------------------------- 2/2 [customtkinter]



##### Petite introduction pour le set-up des couleurs/thèmes

###### Couleurs

###### Toutes les couleurs des widgets peuvent être personnalisées.

"bg_color" correspond uniquement à la couleur derrière le widget, et elle n’est visible que si le widget possède des coins arrondis.

La couleur principale d’un widget est appelée :
"fg_color"

Types de couleurs possibles

Les couleurs peuvent être définies de trois façons :

1. Nom de couleur : "red"
2. Code hexadécimal :  "#FF0000" 
3. Couleur double (mode clair / sombre) : ("red", "darkred")

###### Mode clair / sombre

On utilise un tuple de couleurs :
("red", "darkred")

==> Le widget choisira automatiquement :
"red" en mode clair
"darkred" en mode sombre

Si on utilise une seule couleur :

"red"

==>Cette couleur sera utilisée dans tous les modes (clair et sombre).



###### Thème


Par défaut, toutes les couleurs sont définies par un thème de couleur.
Actuellement, trois thèmes sont disponibles :


"blue" (thème par défaut)


"dark-blue"


"green"

==>Tous ces thèmes utilisent des couleurs en tuple, ce qui permet une adaptation automatique :
##### mode clair OU mode sombre


##### Définir un thème: 

On peut définir le thème au début de notre programme :
ctk.set_default_color_theme("dark-blue")


##### Fenêtre principale

L’application repose sur une fenêtre principale créée avec la classe CTk.
Cette fenêtre constitue le "conteneur global" dans lequel tous les éléments graphiques sont affichés.

Ses propriétés principales sont :

1. La taille (geometry)
2. Le titre (title)
3. Le mode d’apparence (clair/sombre)

In [6]:
import customtkinter as ctk

ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("dark-blue")

app = ctk.CTk()
app.geometry("900x600")
app.title("PickMyMovie")

app.mainloop()

dans cette cellule on retrouve également ce dont on a parlé dans la petite introduction sur les couleurs et thèmes.

##### Gestion des pages avec les Frames

Les frames sont des "conteneurs graphiques" permettant d’organiser l’interface en différentes sections.

Dans notre application, chaque étape du parcours utilisateur est représentée par un frame distinct.

Pour naviguer entre les frames, nous utilisons une fonction qui met au premier plan le frame souhaité (tkraise).
Cette technique permet de simuler un système de navigation multi-pages sans recréer de nouvelles fenêtres. (on reste toujours dans la fenêtre principale que nous avons créé précédemment)

In [7]:
container = ctk.CTkFrame(app)
container.pack(fill="both", expand=True)

frame1 = ctk.CTkFrame(container)
frame2 = ctk.CTkFrame(container)

for frame in (frame1, frame2):
    frame.grid(row=0, column=0, sticky="nsew")

TclError: can't invoke "frame" command: application has been destroyed

On utilise des CTkFrame pour créer différentes pages (comme un site web).

##### Navigation entre les pages

On change de page avec tkraise(). (comme dit précedemment)


In [9]:
def show_frame(frame):
    frame.tkraise()

# Exemple bouton
ctk.CTkButton(frame1, text="Aller page 2",
              command=lambda: show_frame(frame2)).pack()

NameError: name 'frame1' is not defined

##### Les widgets utilisés

##### 1. Labels

Les labels (CTkLabel) sont utilisés pour afficher du texte à l’écran, comme les titres ou les instructions destinées à l’utilisateur.

In [8]:
ctk.CTkLabel(
    frame1,
    text="Bienvenue présente toi !",
    font=("Arial", 24)
).pack(pady=20)

NameError: name 'frame1' is not defined

##### 2. Boutons

Les boutons (CTkButton) permettent de déclencher des actions lorsque l’utilisateur clique dessus.

Chaque bouton est associé à une fonction (command) qui définit le comportement à exécuter (navigation, validation, etc...).

In [10]:
ctk.CTkButton(frame1, text="Suivant", command=go_to_movies, fg_color="red").pack(pady=20)

NameError: name 'frame1' is not defined

##### 3. Champs de saisie

Les champs de saisie (CTkEntry) permettent à l’utilisateur d’entrer du texte, comme son nom ou les films qu’il a déjà vus.

Les données saisies sont récupérées via la méthode .get().

In [ ]:
f2_title = ctk.CTkLabel(frame2, text="Film 1/5 — Tape un titre", font=("Arial", 20))
f2_title.pack(pady=20)

f2_entry = ctk.CTkEntry(frame2, placeholder_text="Ex: Inception", width=400)
f2_entry.pack(pady=10)

f2_feedback = ctk.CTkLabel(frame2, text="", font=("Arial", 12), text_color="orange")
f2_feedback.pack(pady=5)

f2_suggestions_frame = ctk.CTkFrame(frame2, fg_color="transparent")
f2_suggestions_frame.pack(pady=5)

f2_added_frame = ctk.CTkFrame(frame2, fg_color="transparent")
f2_added_frame.pack(pady=10)

f2_next_btn = ctk.CTkButton(frame2, text="Suivant", command=lambda: go_to_profile(), fg_color="red", state="disabled")
f2_next_btn.pack(pady=20)

On retrouve ici les labels, boutons et champs de saisies.

##### 4. Cases à cocher

Les cases à cocher (CTkCheckBox) permettent de sélectionner plusieurs options.

Elles sont utilisées dans notre application pour permettre à l’utilisateur de choisir plusieurs films recommandés.

In [ ]:
ctk.CTkLabel(frame3, text="Prendre en compte ton profil ?", font=("Arial", 20)).pack(pady=30)
ctk.CTkCheckBox(frame3, text="Oui, utiliser mon historique", variable=use_profile).pack(pady=10)

Permet de sélectionner plusieurs films, ou également utilisé pour le choix de l'utilisation du profil ou non.

In [ ]:
for idx, row in top100_df.iterrows():
        var = ctk.BooleanVar()
        cb = ctk.CTkCheckBox(
            scroll,
            text=row["title"],
            variable=var,
            command=toggle_limit
        )
        cb.pack(anchor="w", padx=10, pady=5)
        rec_vars.append((row["title"], var, row["movieId"]))

    show_frame(frame5)



##### Gestion des variables interactives

L’interface utilise des variables spécifiques (StringVar, BooleanVar) permettant de stocker et synchroniser les données entre les widgets et le programme.

Ces variables facilitent la récupération des choix de l’utilisateur en temps réel.

Exemple :

StringVar → texte (nom, mood)
BooleanVar → vrai/faux (checkbox)

In [12]:
username = ctk.StringVar()
movies_list = []
use_profile = ctk.BooleanVar()
selected_mood = ctk.StringVar(value="")
rec_vars = []

RuntimeError: Too early to create variable: no default root window

##### Scrollable Frame (liste longue)
Permet d’avoir une liste scrollable (important pour trouver les 5 films dans la liste).

In [ ]:
for widget in scroll.winfo_children():
        widget.destroy()
    rec_vars.clear()

    for idx, row in top100_df.iterrows():
        var = ctk.BooleanVar()
        cb = ctk.CTkCheckBox(
            scroll,
            text=row["title"],
            variable=var,
            command=toggle_limit
        )
        cb.pack(anchor="w", padx=10, pady=5)
        rec_vars.append((row["title"], var, row["movieId"]))

    show_frame(frame5)


ctk.CTkButton(frame4, text="Voir recommandations", command=go_to_reco, fg_color="red").pack(pady=40)

ctk.CTkLabel(frame5, text="Coche jusqu'à 5 films que tu aimes !", font=("Arial", 20)).pack(pady=10)
scroll = ctk.CTkScrollableFrame(frame5, width=700, height=350)
scroll.pack(pady=10)
rec_vars = []

##### Limiter le nombre de choix (max 5)

On empêche l’utilisateur de cocher plus de 5 films.

In [13]:
def toggle_limit():
    count = sum(var.get() for _, var, _ in rec_vars)
    if count > 5:
        for name, var, mid in reversed(rec_vars):
            if var.get():
                var.set(False)
                break

##### Validation des entrées

On empêche de passer à la suite si les champs ne sont pas remplis.

In [14]:
def go_to_movies():
    if entry_name.get().strip() == "":
        return

##### Affichage du résultat final

On affiche les recommandations finales.

In [ ]:
top5 = final_scores.sort_values(ascending=False).head(5)
    top5_titles = []
    for tc in top5.index:
        match = movies[movies["title_clean"] == tc]
        top5_titles.append(match.iloc[0]["title"] if not match.empty else tc)

    medals = ["🥇", "🥈", "🥉", "4.", "5."]
    sizes = [20, 18, 16, 14, 14]
    for widget in frame6.winfo_children():
        widget.destroy()

    ctk.CTkLabel(frame6, text="Top 5 meilleurs films recommandés pour toi", font=("Arial", 22)).pack(pady=20)
    for i, title in enumerate(top5_titles):
        bold = "bold" if i == 0 else "normal"
        ctk.CTkLabel(frame6, text=f"{medals[i]} {title}", font=("Arial", sizes[i], bold)).pack(pady=10)

    show_frame(frame6)

##### Navigation et logique utilisateur

L’interface suit un flux "logique" guidant l’utilisateur étape par étape.

À chaque étape, une validation est effectuée afin de garantir que les informations nécessaires sont correctement saisies avant de passer à l’étape suivante. 

Cela permet :

on évite les erreurs
On s'assure de la cohérence des données
==>On améliore l’expérience utilisateur

(on évite comme cela les erreurs, qu'elles soient de saisies ou bien du fait qu'un film n'existe pas etc)


##### Validation des entrées

On va vérifier :

1. nom non vide
2. nombre suffisant de films saisis
3. sélection d’un mood
4. sélection d’au moins un film recommandé

##### Choix design

Nous avons choisi le thème sombre pour le confort visuel, les boutons sont rouges pour rappeler les fauteuils traditionels des cinémas.


##### Bilan de l'interface 

L’interface développée permet une interaction fluide et structurée entre l’utilisateur et le système de recommandation. Ce qui donne un ressenti d'application moderne.

Grâce à l’utilisation de CustomTkinter, nous avons pu concevoir une interface moderne et intuitive adaptée à une application de recommandation de films. 

#### Utilisation des 3 modèles

Le pipeline combine les 3 modèles en séquence, en entonnoir comme on vous le montre ci-dessous :

Modèle des moods : FastText. À partir des titres des 5 films aimés et du mood choisi, on extrait des mots-clés que l'on vectorise via FastText en vecteurs de dimension 300. On calcule leur similarité cosinus avec tous les tags du genome pour identifier les plus proches. On retourne les 100 films ayant les scores de relevance moyens les plus élevés sur ces tags. Nous avions au tout début choisi de n'en afficher que 20 mais, au vu du nombre de film qui existe, le risque que l'utilisateur n'en ai vu aucun est trop grand.

Modèle 3 des similarités de genre : Similarité de Jaccard
Pour chaque film présent dans les 100 présentés par le modèles des moods, nous avons fait en sorte que la machine calcule une similarité pondérée par rapport aux films aimés par ce même utilisateur selon 3 critères :
- Genres Jaccard pour retrancher les films ayant un genre proche : poids 0.5
- Note moyenne pour retrancher les films ayant des notes proches : poids 0.3
- Année de sortie afin de retrancher les films ayant une proximité temporelle: poids 0.2

Modèle collaboratif : Similarité Utilisateurs
On créé au début de notre questionnaire un utilisateur, ce dernier indique les 5 derniers films qu'il a aimé, nous partons du principe que les notes sont de 5 pour ces films là. La similarité cosinus est calculée face aux 3 000 utilisateurs les plus actifs, c'est à dire ce qui ont notés les plus de film. Ce modèle est optionnel : l'utilisateur choisit de l'activer ou non lorsque nous commençons le questionnaire.

Pour le score final, les scores Jaccard et collaboratif sont normalisés entre 0 et 1, et nous avons décidé qu'ils soient combinés à 60% Jaccard et 40% Collaboratif

In [2]:
if use_collab:
    final_scores = 0.6 * jaccard_scores + 0.4 * collab_scores_norm
else:
    final_scores = jaccard_scores

best_title_clean = final_scores.idxmax()

NameError: name 'use_collab' is not defined

#### Résultats

##### modèle 1 : Modèle collaboratif basé sur les utilisateurs

Voici le résultat de user_movie_matrix. On peut y voir les notes des utilisateurs pour chaque film. Lorsqu'il y a 0.0, cela veut dire que l'utilisateur n'a pas vu le film. On peut alors éliminer les films qu'il a vu dans les propositions.

<img src="images/user_movie_matrix.png" width="500">


Sur cette deuxième image, on voit les similarités de goûts entre les utilisateurs. Ainsi, si 2 utilisateurs ont des goûts très proches, et que utilisateur 1 n'a pas vu un des films apprécié par utilisateur 2, notre code va le repérer et va pouvoir lui conseiller ce film.

<img src="images/similarités_matrix.png" width="500">

##### modèle 2 : Modèle de recherche sémantique basée sur l'humeur de l'utilisateur

Pour vérifier le fonctionnement de ce modèle, nous avons pris 2 exemples : 

    - appel du modèle avec des tags qui se ressemblent : robot, technology, utopia
    - appel du modèle avec des tags qui sont différents : romantic, poker, rabbits

<img src="images/similar_tags_modele_2_tags_proches.png" width="500">
<img src="images/similar_tags_modele_2_tags_differents.png" width="500">

On constate ici le comportement de FastText : il récupère bien les tags les plus similaires au tag d'entrée. De plus, il garde les tags donnés en entrée s'ils sont dans la base de donnée. On voit que les notes sont assez hautes et que les tags trouvés ont un lien évident avec les tags de départ (exemple : pigs est bien en lien avec rabbits car c'est un animal).

<img src="images/top_movies_modele_2_tags_proches.png" width="500">
<img src="images/top_movies_modele_2_tags_differents.png" width="500">

Ensuite, ces graphes confirment bien que plus on donne des tags qui n'ont pas de lien entre eux en entrée, plus les notes de similarités des films donnés par le modèle sont faibles.

<img src="images/heatmap_modele_2_tags_proches.png" width="600">
<img src="images/heatmap_modele_2_tags_differents.png" width="600">

Enfin, on constate ici que :

    - Pour le heatmap 1, tous les films proposés par le modèle ont des notes assez hautes pour tous les tags qui ont été trouvés par le modèle suite à l'appel de FastText
    - Pour le heatmap 2, certains films sont très liés à certains tags, mais certains tags n'est similaire à aucun film.
    
Cela montre le comportement de notre modèle : Lorsqu'on donne des tags assez différents en entrée, il va trouver des films qui ont de très grosses notes pour 1 ou 2 tags, et donc va avoir une moyenne assez haute , malgré ses autres notes qui sont basses. 

##### Modèle des similarités de genre : Similarité de Jaccard

Pour vérifier le fonctionnement de ce modèle, nous avons fait 2 exemples :
    
    - Un test dans lequel on ne donne que des films qui se ressemblent : ici nous avons donné 3 films de la saga Star Wars
    - Un test dans lequel on ne donne que des films différents : toy story, casino, Anne Frank

<img src="images/modele_3_films_proches.png" width="600">
<img src="images/modele_3_films_differents.png" width="600">

On constate bien ici que :

    - Les films proposés au premier test sont tous très proches et dans les mêmes thèmes. Les notes sont toutes au dessus de 90% de similarité
    - Les films proposés dans le deuxième test sont très en lien avec 1 des films donnés en entrée, mais ont des notes très basses pour les autres. L'échelle des notes descent jusqu'à 50% de similarité.

#### Améliorations envisagés

Il serait intéressant de faire en sorte d'avoir un historique utilisateur qui stock au fur et à mesure les films que celui-ci a vu et aime. Ainsi, on viendrait stocker les anciennes entrées de l'utilisateur à chaque nouvelle utilisation, ce qui permettrait d'avoir un profil qui évolue et s'enrichie au fil des utilisations. Toujours dans le but d'affiner de plus en plus la recommandation. D'autres parts, nettoyer d'avantage le dataset ou alors trouver un dataset avec des films disposants de plus de notes serait plus pertinent. Il serait aussi intéressant de pouvoir régulièrement mettre à jour notre dataset, pour avoir les derniers films sortis dans celui-ci. En effet, le dataset n'est pas totalement à jour, ce qui empêche par exemple d'avoir des films tels que "The drama" ou encore "Zootopie 2". Il serait donc intéréssent de mettre à jour cela tous les mois par exemple.  

#### Ouverture

On pourrait également proposer des films totalement différents mais qui pourrait permettre à l'utilisateur de diversifier ses propositions et recommandations, et ainsi ne pas toujours proposer des films trop similaires. On appellerai cela "la suggestion du chef" ou un nom du même genre, à la manière des restaurants qui proposent un plat en vedette on pourrait avoir un film en vedette qui change chaque semaine ou en fonction des catégories de films recommandés (pour être différent de ceux déjà proposés)

### Liens Github
https://github.com/zeliedelloye-coder/Pick_My_Movie_GRP1_Z-lieDelloye-LilianeAroule-ThibaultRoyer)

### Conclusion

Notre projet avait pour objectif de concevoir un système de recommandation de films combinant une interface graphique interactive et des techniques d’intelligence artificielle. À travers le développement de notre système *PickMyMovie*, nous avons pu mettre en œuvre plusieurs concepts clés, allant de la manipulation de données à l’intégration de modèles de recommandation.

Dans un premier temps, nous avons exploité le dataset "MovieLens" afin de construire une base de données riche contenant des informations sur les films, les notes des utilisateurs et des descripteurs sémantiques (tags). 

Nous avons ensuite implémenté un système de recommandation hybride reposant sur deux approches complémentaires : une méthode basée sur le contenu (similarité de Jaccard entre films) et une méthode collaborative exploitant les préférences d’utilisateurs similaires. 

Par la suite, nous avons intégré un modèle de traitement du langage naturel (FastText) permettant d’interpréter les préférences exprimées par l’utilisateur sous forme de mots-clés (mood), rendant l’expérience plus intuitive et personnalisée. 

Notre interface graphique développée avec CustomTkinter joue également un rôle central dans le projet. Elle permet une interaction fluide avec l’utilisateur à travers plusieurs étapes : saisie du nom de l'utilisateur et de ses films préférés, définition du profil, expression du mood et sélection des recommandations. L’affichage final met en avant les résultats de manière claire et hiérarchisée. 

Cependant, certaines limites subsistent. Le temps de chargement peut être important en raison de la taille des données et du modèle utilisé. De plus, l’absence d’éléments visuels comme les affiches de films peut rendre l’interface moins immersive.

En perspective, plusieurs améliorations pourraient être envisagées, telles que l’ajout d’images de films, l’optimisation des performances, ou encore le déploiement de l’application sous forme web pour une accessibilité élargie. 

En conclusion, ce projet nous a permis de mobiliser des compétences variées en programmation, en traitement de données et en intelligence artificielle, tout en développant une "application" concrète et interactive répondant à un besoin réel de recommandation personnalisée. 
